# Repeatable job

`RepeatableJob` wraps an `ns.Job` and submits it again when an attempt is interrupted, for example when a spot VM is preempted. It is a minimal prototype: one class, about twenty lines.

Rescheduling only helps when the job can continue where it stopped. The worker in this guide, [`resumable_worker.py`](resumable_worker.py), follows three rules:

1. On start, load the last checkpoint from storage that outlives the job (here, a mounted bucket).
2. On SIGTERM, save a checkpoint at the next step boundary and exit non-zero, so the attempt ends `FAILED`.
3. Exit 0 only when the work is done.

Preemption is simulated: coreutils `timeout` sends SIGTERM to the worker after 60 seconds. Training takes 150 one-second steps, so it needs three attempts.

**Before you start**

- `uv sync` in the repo root, then start Jupyter from this directory, for example `uv run --with jupyter jupyter lab examples/signals`.
- A Nebius CLI profile (`nebius profile create`) and a bucket you can mount read-write. Set `BUCKET` below.
- The jobs run on a CPU machine (`cpu-d3`, `2vcpu-8gb`). **Running section 3 starts up to five billable jobs.**

In [ ]:
import nebius_serverless as ns

ns.diagnostics()

In [ ]:
from pathlib import Path
from uuid import uuid4

BUCKET = "ez-bucket-1"  # any bucket you can mount read-write
CKPT_DIR = f"/ckpt/repeatable-job/{uuid4().hex[:8]}"  # fresh per notebook run
PREEMPT_AFTER = 60  # simulated preemption: SIGTERM after this many seconds
TOTAL_STEPS = 150  # one step is one second
WORKER = Path("resumable_worker.py").resolve()  # uploaded into each attempt

CKPT_DIR

## 1. The wrapper

Each attempt is a new job created from the same recipe, with its own id and logs. The wrapper only decides whether to submit again. Progress lives in the checkpoint, not in the wrapper.

`retry_on` lists the states that mean "interrupted, try again". `CANCELLED` is left out, so cancelling an attempt by hand stops the loop instead of starting another job.

`Job(retries=N)` asks Nebius to restart the job. `RepeatableJob` does it from the client: the loop is plain Python, so it can change the recipe between attempts, for example to fall back from spot to on-demand.

In [ ]:
class RepeatableJob:
    """Submit a job again until one attempt completes."""

    def __init__(
        self,
        job: ns.Job,
        *,
        max_attempts: int = 3,
        retry_on: tuple[str, ...] = ("FAILED", "ERROR"),
    ) -> None:
        self.job = job
        self.max_attempts = max_attempts
        self.retry_on = retry_on
        self.runs: list[ns.Run] = []  # one Run per attempt

    async def run_async(self) -> ns.Run:
        for attempt in range(1, self.max_attempts + 1):
            run = await self.job.create_async()
            self.runs.append(run)
            await run.wait_async()
            if run.status not in self.retry_on:
                return run  # COMPLETED, or a state that should not repeat
            print(f"attempt {attempt}/{self.max_attempts} ended {run.status}")
        raise RuntimeError(
            f"no attempt completed in {self.max_attempts} tries"
        )

## 2. Describe the job

- `volumes` mounts the bucket at `/ckpt`, so the checkpoint survives between attempts.
- `inject_files` uploads the worker into the container. `WORKER` is resolved from the notebook directory.
- `timeout --preserve-status` sends SIGTERM after `PREEMPT_AFTER` seconds and keeps the worker's exit code: 143 after SIGTERM, 0 when training is complete. Drop the `timeout` prefix for a real workload.

In [ ]:
job = ns.Job(
    [
        "timeout",
        "--preserve-status",
        "--signal=TERM",
        "--kill-after=30",
        str(PREEMPT_AFTER),
        "python",
        "-u",
        "/app/resumable_worker.py",
    ],
    image="python:3.12-slim",
    timeout="1h",
    volumes=[f"s3://{BUCKET}:/ckpt:rw"],
    inject_files=[f"{WORKER}:/app/resumable_worker.py"],
    env={"CKPT_DIR": CKPT_DIR, "TOTAL_STEPS": str(TOTAL_STEPS)},
    name="repeatable-job",
)

## 3. Run until complete

Expect three attempts: the first two end `FAILED` after the simulated preemption, the third completes. Each attempt waits a minute or two for a machine.

Interrupting this cell stops the loop but not the current attempt. Cancel it with `await repeatable.runs[-1].cancel_async()`.

In [ ]:
repeatable = RepeatableJob(job, max_attempts=5)
run = await repeatable.run_async()
run.display()

## 4. Check the attempts

Every attempt should start at the step where the previous one saved its checkpoint.

In [ ]:
for attempt, r in enumerate(repeatable.runs, start=1):
    print(f"attempt {attempt}: {r.id} {r.status}")
    !nebius ai job logs {r.id} | grep -E "starting|SIGTERM|done"